In [1]:
import pandas as pd
import seaborn as sns
import xgboost as xgb
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/LR_MV_sintetic_07_05_2026.parquet')
#output_eadms_sintetic_07_05_2026.parquet')

In [6]:
df.columns.to_list()

['co_ibge',
 'year_week',
 'atend_ivas',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'mem_surge_01_replicate_0',
 'mem_surge_01_replicate_0_correct_with_consec',
 'mem_surge_01_replicate_1',
 'mem_surge_01_replicate_1_correct_with_consec',
 'mem_surge_01_replicate_2',
 'mem_surge_01_replicate_2_correct_with_consec',
 'mem_surge_01_replicate_3',
 'mem_surge_01_replicate_3_correct_with_consec',
 'mem_surge_01_replicate_4',
 'mem_surge_01_replicate_4_correct_with_consec',
 'mem_surge_01_replicate_5',
 'mem_surge_01_replicate_5_correct_with_consec',
 'mem_surge_01_replicate_6',
 'mem_surge_01_replicate_6_correct_with_consec',
 'mem_surge_01_replicate_7',
 'mem_surge_01_replicate_7_correct_with_consec',
 'mem_surge_01_replicate_8',
 'mem_surge_01_replicate_8_correct_with_consec',
 'mem_surge_01_replicate_9',
 'mem_surge_01_replicate_9_correct_with_consec',
 'mem_surge_01_replicate_10',
 'mem_surge_01_replicate_10_correct_with_consec',
 'mem_surge_01_replicate_11',
 

In [8]:
# --- Função para criar Lag Features (Janelas de Atraso) ---
def criar_lag_features(df, colunas_sinais, lags=[1, 2, 3]):
    """
    Cria colunas de atraso (lags) para dar contexto histórico ao modelo.
    Ex: Se lag=1, cria uma coluna com o valor da semana passada.
    """
    df_lagged = df.copy()
    
    # Ordena por tempo ('year_week') e local ('co_ibge7') para garantir que o shift pegue a semana anterior correta
    if 'co_ibge7' in df.columns:
        df_lagged = df_lagged.sort_values(by=['co_ibge7', 'year_week'])
    else:
        df_lagged = df_lagged.sort_values(by=['year_week'])

    new_features = []
    
    for col in colunas_sinais:
        for lag in lags:
            nome_nova_coluna = f"{col}_lag{lag}"
            # O shift(lag) empurra os valores para baixo
            if 'co_ibge7' in df.columns:
                # Agrupa por município para não pegar dado de uma cidade e jogar na outra
                df_lagged[nome_nova_coluna] = df_lagged.groupby('co_ibge7')[col].shift(lag)
            else:
                df_lagged[nome_nova_coluna] = df_lagged[col].shift(lag)
            
            new_features.append(nome_nova_coluna)
    
    # Trata as primeiras linhas que ficaram vazias (NaN) por causa do shift
    for col in new_features:
        print(f"QTD Valores NaN na coluna '{col}' : '{df_lagged[col].isnull().sum()}' ")
        if df_lagged[col].isnull().any():
            moda = df_lagged[col].mode()[0] # .mode() retorna uma série, pegamos o primeiro valor
            df_lagged[col].fillna(moda, inplace=True)
            print(f"Valores NaN na coluna '{col}' preenchidos com a moda: '{int(moda)}' ")
    #df_lagged = df_lagged.dropna()

    
    return df_lagged, new_features


In [9]:
def run_xgb(data, replicates=range(32), lags=[1, 2, 3]):

    results = []

    for rep in replicates:

        print(f"\n===== Processing replicate {rep} =====")

        columns_sinais = [
            f'C2_alarms_{rep}',
            f'sinal_evi_replicate_{rep}',
            f'EWS_ISF_replicate_{rep}',
            f'EWS_LOF_replicate_{rep}',
            f'EWS_OCSVM_replicate_{rep}',
            f'EWS_COPOD_replicate_{rep}',
            f'EWS_Rt_replicate_{rep}'
        ]

        col_surge = f'mem_surge_01_replicate_{rep}'
        col_surge_consec = (
            f'mem_surge_01_replicate_{rep}_correct_with_consec'
        )

        # Create lagged features
        df_com_lags, nomes_features_lags = criar_lag_features(
            data,
            columns_sinais,
            lags=lags
        )

        features_para_treino = (
            columns_sinais + nomes_features_lags
        )

        X = df_com_lags[features_para_treino]
        y = df_com_lags[col_surge_consec]

        # Skip if only one class
        if y.nunique() < 2:
            print(f"Skipping replicate {rep}: only one class.")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.3,
            random_state=42,
            stratify=y
        )

        counter = (
            len(y_train[y_train == 0]) /
            max(len(y_train[y_train == 1]), 1)
        )

        model = xgb.XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42,
            eval_metric='logloss',
            scale_pos_weight=counter,
            max_delta_step=1
        )

        model.fit(X_train, y_train)

        print("Preparing production data...")

        df_prod = df_com_lags[
            (df_com_lags['year_week'] >= '2022-42') &
            (df_com_lags['year_week'] <= '2025-32')
        ].copy()

        X_prod = df_prod[features_para_treino]

        print("Generating probabilities...")

        df_prod[f'prob_ensemble_xgb_rep_{rep}'] = (
            model.predict_proba(X_prod)[:, 1]
        )

        df_prod[f'signal_ensemble_xgb50_rep_{rep}'] = (
            df_prod[f'prob_ensemble_xgb_rep_{rep}'] >= 0.50
        ).astype(int)

        keep_cols = [
            'co_ibge',
            'year_week',
            f'prob_ensemble_xgb_rep_{rep}',
            f'signal_ensemble_xgb50_rep_{rep}'
        ]

        results.append(df_prod[keep_cols])

    # Merge results
    final_res = data.copy()

    for df_rep in results:

        final_res = final_res.merge(
            df_rep,
            on=['co_ibge', 'year_week'],
            how='left'
        )

    return final_res

In [10]:
final_df = run_xgb(df)


===== Processing replicate 0 =====
QTD Valores NaN na coluna 'C2_alarms_0_lag1' : '1' 
Valores NaN na coluna 'C2_alarms_0_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'C2_alarms_0_lag2' : '2' 
Valores NaN na coluna 'C2_alarms_0_lag2' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'C2_alarms_0_lag3' : '3' 
Valores NaN na coluna 'C2_alarms_0_lag3' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_replicate_0_lag1' : '1' 
Valores NaN na coluna 'sinal_evi_replicate_0_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_replicate_0_lag2' : '2' 
Valores NaN na coluna 'sinal_evi_replicate_0_lag2' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'sinal_evi_replicate_0_lag3' : '3' 
Valores NaN na coluna 'sinal_evi_replicate_0_lag3' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'EWS_ISF_replicate_0_lag1' : '1' 
Valores NaN na coluna 'EWS_ISF_replicate_0_lag1' preenchidos com a moda: '0' 
QTD Valores NaN na coluna 'EWS_ISF

In [21]:
for rep in range(32):
    keep_cols = [
            'co_ibge',
            'year_week',
            f'prob_ensemble_xgb_rep_{rep}',
            f'signal_ensemble_xgb50_rep_{rep}'
        ]
    
    print(rep, final_df[keep_cols].isnull().sum().to_list())

0 [0, 0, 0, 0]
1 [0, 0, 0, 0]
2 [0, 0, 0, 0]
3 [0, 0, 0, 0]
4 [0, 0, 0, 0]
5 [0, 0, 0, 0]
6 [0, 0, 0, 0]
7 [0, 0, 0, 0]
8 [0, 0, 0, 0]
9 [0, 0, 0, 0]
10 [0, 0, 0, 0]
11 [0, 0, 0, 0]
12 [0, 0, 0, 0]
13 [0, 0, 0, 0]
14 [0, 0, 0, 0]
15 [0, 0, 0, 0]
16 [0, 0, 0, 0]
17 [0, 0, 0, 0]
18 [0, 0, 0, 0]
19 [0, 0, 0, 0]
20 [0, 0, 0, 0]
21 [0, 0, 0, 0]
22 [0, 0, 0, 0]
23 [0, 0, 0, 0]
24 [0, 0, 0, 0]
25 [0, 0, 0, 0]
26 [0, 0, 0, 0]
27 [0, 0, 0, 0]
28 [0, 0, 0, 0]
29 [0, 0, 0, 0]
30 [0, 0, 0, 0]
31 [0, 0, 0, 0]


In [27]:
#dta =  final_df.isnull().sum().reset_index()

In [28]:
#dta[dta[0] >= 1]

In [33]:
final_df.columns.to_list()

['co_ibge',
 'year_week',
 'atend_ivas',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'mem_surge_01_replicate_0',
 'mem_surge_01_replicate_0_correct_with_consec',
 'mem_surge_01_replicate_1',
 'mem_surge_01_replicate_1_correct_with_consec',
 'mem_surge_01_replicate_2',
 'mem_surge_01_replicate_2_correct_with_consec',
 'mem_surge_01_replicate_3',
 'mem_surge_01_replicate_3_correct_with_consec',
 'mem_surge_01_replicate_4',
 'mem_surge_01_replicate_4_correct_with_consec',
 'mem_surge_01_replicate_5',
 'mem_surge_01_replicate_5_correct_with_consec',
 'mem_surge_01_replicate_6',
 'mem_surge_01_replicate_6_correct_with_consec',
 'mem_surge_01_replicate_7',
 'mem_surge_01_replicate_7_correct_with_consec',
 'mem_surge_01_replicate_8',
 'mem_surge_01_replicate_8_correct_with_consec',
 'mem_surge_01_replicate_9',
 'mem_surge_01_replicate_9_correct_with_consec',
 'mem_surge_01_replicate_10',
 'mem_surge_01_replicate_10_correct_with_consec',
 'mem_surge_01_replicate_11',
 

In [32]:
from pathlib import Path
from datetime import datetime

out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")

fname = f"LR_MV_xgb_sintetic_{datetime.now():%d_%m_%Y}.parquet"

final_df.to_parquet(out_dir / fname)